## Gold — conformed dimensions.

 Dimension strategy:
   dim_patient, dim_doctor, dim_department, dim_hospital, dim_bed  -> SCD2
     These change in ways history matters for. A doctor who moved
     department in 2025 must not have their 2024 admissions reattributed.
   dim_diagnosis, dim_lab_test, dim_medication, dim_insurance_provider -> SCD1
     The code is the identity. A description edit is not a new fact about
     the world, and retired codes stay so historical facts still resolve.
   dim_date, dim_time -> static

 Every dimension is seeded with -1 unknown, -2 not applicable, -3 late
 arriving. A fact that cannot resolve a lookup points at these rather than
 being dropped, so Silver-to-Gold counts reconcile and the gap surfaces as
 a measure instead of a silent undercount.


In [ ]:

#batch_id = "GOLD_MANUAL"
SILVER = "lh_silver.dbo"
BRONZE = "lh_bronze.dbo"
#date_start = "2023-01-01"
#date_end = "2027-12-31"

In [ ]:
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType

run_ts = datetime.now(timezone.utc)
HIGH_DATE = "9999-12-31 23:59:59"
summary = []

print(f"Gold dimension load {batch_id}")

## Helpers

In [ ]:
def add_hash(df, cols):
    """Deterministic hash over the tracked columns.

    Trimmed and upper-cased so a cosmetic source change does not create a
    spurious SCD2 version. Nulls get a sentinel so (null,'A') and ('A',null)
    hash differently.
    """
    return df.withColumn("row_hash", F.sha2(F.concat_ws("||", *[
        F.coalesce(F.upper(F.trim(F.col(c).cast(StringType()))), F.lit("<NULL>"))
        for c in cols]), 256))


def seed_unknown(table, key_col, bk_col, extra=None):
    """Insert the -1/-2/-3 members if absent."""
    if spark.table(table).filter(F.col(key_col) < 0).count() >= 3:
        return
    rows = [(-1, "UNKNOWN"), (-2, "N/A"), (-3, "LATE_ARRIVING")]
    df = spark.createDataFrame(rows, f"{key_col} long, {bk_col} string")
    for c, v in (extra or {}).items():
        df = df.withColumn(c, F.lit(v))
    df = (df.withColumn("row_hash", F.lit("SPECIAL"))
            .withColumn("updated_ts", F.lit(run_ts)))
    if "effective_from_ts" in spark.table(table).columns:
        df = (df.withColumn("effective_from_ts", F.lit("1900-01-01").cast("timestamp"))
                .withColumn("effective_to_ts", F.lit(HIGH_DATE).cast("timestamp"))
                .withColumn("is_current", F.lit(True))
                .withColumn("created_ts", F.lit(run_ts)))
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)


def scd2_merge(src, table, business_key, tracked, key_col):
    """Two-phase SCD Type 2 upsert.

    Phase 1 expires the current row whose hash differs. Phase 2 inserts the
    replacement plus any brand-new members. Delta MERGE cannot both update a
    row and insert its successor in one pass, which is why this is split.
    """
    src = add_hash(src, tracked)

    if not spark.catalog.tableExists(table):
        w = Window.orderBy(F.col(business_key))
        out = (src.withColumn(key_col, F.row_number().over(w).cast("long"))
                  # An initial load is not "these members came into existence
                  # today" — it is a backfill of members that already existed.
                  # Dating them from run_ts makes every historical fact fall
                  # outside their validity window, and every temporal lookup
                  # silently resolves to the unknown member.
                  .withColumn("effective_from_ts", F.lit("1900-01-01").cast("timestamp"))
                  .withColumn("effective_to_ts", F.lit(HIGH_DATE).cast("timestamp"))
                  .withColumn("is_current", F.lit(True))
                  .withColumn("created_ts", F.lit(run_ts))
                  .withColumn("updated_ts", F.lit(run_ts)))
        out.write.format("delta").mode("overwrite") \
           .option("overwriteSchema", "true").saveAsTable(table)
        n = spark.table(table).count()
        summary.append((table, n, "initial load"))
        print(f"  {table:<28} {n:>8,} rows (initial)")
        return

    tgt = DeltaTable.forName(spark, table)
    (tgt.alias("t")
        .merge(src.alias("s"),
               f"t.{business_key} = s.{business_key} AND t.is_current = true")
        .whenMatchedUpdate(condition="t.row_hash <> s.row_hash",
                           set={"effective_to_ts": F.lit(run_ts),
                                "is_current": F.lit(False),
                                "updated_ts": F.lit(run_ts)})
        .execute())

    current = (spark.table(table).filter(F.col("is_current"))
               .select(F.col(business_key).alias("_bk"), F.col("row_hash").alias("_h")))
    to_insert = (src.join(current, src[business_key] == F.col("_bk"), "left")
                    .filter(F.col("_bk").isNull() | (F.col("_h") != F.col("row_hash")))
                    .drop("_bk", "_h"))

    n_new = to_insert.count()
    if n_new:
        max_key = spark.table(table).agg(
            F.coalesce(F.max(key_col), F.lit(0)).alias("m")).collect()[0]["m"]
        w = Window.orderBy(F.col(business_key))
        to_insert = (to_insert
            .withColumn(key_col, (F.row_number().over(w) + F.lit(max_key)).cast("long"))
            .withColumn("effective_from_ts", F.lit(run_ts))
            .withColumn("effective_to_ts", F.lit(HIGH_DATE).cast("timestamp"))
            .withColumn("is_current", F.lit(True))
            .withColumn("created_ts", F.lit(run_ts))
            .withColumn("updated_ts", F.lit(run_ts)))
        to_insert.write.format("delta").mode("append") \
                 .option("mergeSchema", "true").saveAsTable(table)

    n = spark.table(table).count()
    summary.append((table, n, f"{n_new} new versions"))
    print(f"  {table:<28} {n:>8,} rows ({n_new} new versions)")


def scd1_load(src, table, key_col, business_key):
    """Overwrite in place — the code is the identity."""
    w = Window.orderBy(F.col(business_key))
    out = (src.withColumn(key_col, F.row_number().over(w).cast("long"))
              .withColumn("row_hash", F.lit("SCD1"))
              .withColumn("updated_ts", F.lit(run_ts)))
    out.write.format("delta").mode("overwrite") \
       .option("overwriteSchema", "true").saveAsTable(table)
    n = spark.table(table).count()
    summary.append((table, n, "overwrite"))
    print(f"  {table:<28} {n:>8,} rows")



## 1. dim_date

 Fiscal year runs April to March — the Ontario public-sector convention,
 and what every finance report expects.


In [ ]:
if not spark.catalog.tableExists("dim_date"):
    d = spark.sql(f"SELECT explode(sequence(to_date('{date_start}'), "
                  f"to_date('{date_end}'), interval 1 day)) AS calendar_date")
    d = (d
        .withColumn("date_key", F.date_format("calendar_date", "yyyyMMdd").cast("int"))
        .withColumn("day_of_week", F.dayofweek("calendar_date"))
        .withColumn("day_name", F.date_format("calendar_date", "EEEE"))
        .withColumn("is_weekend", F.dayofweek("calendar_date").isin(1, 7))
        .withColumn("day_of_month", F.dayofmonth("calendar_date"))
        .withColumn("month_number", F.month("calendar_date"))
        .withColumn("month_name", F.date_format("calendar_date", "MMMM"))
        .withColumn("month_year_label", F.date_format("calendar_date", "MMM yyyy"))
        .withColumn("quarter_number", F.quarter("calendar_date"))
        .withColumn("calendar_year", F.year("calendar_date"))
        .withColumn("fiscal_year",
                    F.when(F.month("calendar_date") >= 4, F.year("calendar_date"))
                     .otherwise(F.year("calendar_date") - 1))
        .withColumn("fiscal_quarter",
                    F.when(F.month("calendar_date").between(4, 6), 1)
                     .when(F.month("calendar_date").between(7, 9), 2)
                     .when(F.month("calendar_date").between(10, 12), 3)
                     .otherwise(4))
        .withColumn("fiscal_period",
                    F.when(F.month("calendar_date") >= 4, F.month("calendar_date") - 3)
                     .otherwise(F.month("calendar_date") + 9))
        .withColumn("week_start_date", F.date_trunc("week", "calendar_date").cast("date"))
        .withColumn("is_holiday", F.lit(False)))

    # The unknown member. A fact with no date points here rather than
    # defaulting to a real day, which would silently distort every trend it
    # appeared in.
    unk = spark.createDataFrame([(-1,)], "date_key int")
    for c in d.columns:
        if c != "date_key":
            unk = unk.withColumn(c, F.lit(None).cast(d.schema[c].dataType))

    d.unionByName(unk).write.format("delta").mode("overwrite") \
     .option("overwriteSchema", "true").saveAsTable("dim_date")
print(f"  dim_date                     {spark.table('dim_date').count():>8,} rows")

## 2. dim_time

 One row per minute of the day. Separate from dim_date so an ED arrival can
 be sliced by hour without exploding the date dimension to millions of rows.


In [ ]:
if not spark.catalog.tableExists("dim_time"):
    t = spark.sql("SELECT explode(sequence(0, 1439)) AS time_key")
    hour = (F.col("time_key") / 60).cast("int")
    t = (t
        .withColumn("hour_24", hour)
        .withColumn("minute_of_hour", F.col("time_key") % 60)
        .withColumn("time_label",
                    F.concat(F.lpad(hour.cast("string"), 2, "0"), F.lit(":"),
                             F.lpad((F.col("time_key") % 60).cast("string"), 2, "0")))
        # Shifts matter for ED analysis: wait times differ sharply between
        # day, evening and overnight staffing.
        .withColumn("shift_name",
                    F.when(F.col("hour_24").between(7, 14), "Day")
                     .when(F.col("hour_24").between(15, 22), "Evening")
                     .otherwise("Night")))
    unk = spark.createDataFrame([(-1,)], "time_key int")
    for c in t.columns:
        if c != "time_key":
            unk = unk.withColumn(c, F.lit(None).cast(t.schema[c].dataType))
    t.unionByName(unk).write.format("delta").mode("overwrite") \
     .option("overwriteSchema", "true").saveAsTable("dim_time")
print(f"  dim_time                     {spark.table('dim_time').count():>8,} rows")

## 3. dim_patient (SCD2)

 Note what is NOT tracked as Type 2: match_confidence and
 source_record_count change every time the MPI runs, so versioning on them
 would create a new patient row nightly for no analytical benefit. They are
 Type 1 attributes living on a Type 2 dimension.

 No names, no health card, no phone, no full postal code. Those stay in
 silver_patient_pii, restricted to stewards. Gold sees a token and an age
 band.


In [ ]:
pat = (spark.table(f"{SILVER}.silver_patient_golden")
    .select("patient_golden_id", "source_patient_id",
            F.coalesce(F.col("hcn_token"), F.lit("NO_HCN")).alias("patient_token"),
            "birth_year", "age_band", "sex", "fsa",
            F.col("language").alias("preferred_language"),
            "is_deceased", "source_record_count", "match_confidence",
            "contributing_sources"))

scd2_merge(pat, "dim_patient", "patient_golden_id",
           ["sex", "age_band", "fsa", "preferred_language", "is_deceased"],
           "patient_key")
seed_unknown("dim_patient", "patient_key", "patient_golden_id",
             {"age_band": "Unknown", "sex": "UNKNOWN"})

## 4. Facility and staff dimensions (SCD2)

In [ ]:
scd2_merge(
    spark.table(f"{SILVER}.silver_hospital").select(
        "hospital_id", "hospital_name", "facility_type", "region",
        "health_region_code", "city", "province",
        F.col("licensed_beds").cast("int").alias("licensed_beds"),
        (F.col("has_emergency_dept") == "1").alias("has_emergency_dept"),
        (F.col("is_teaching_hospital") == "1").alias("is_teaching_hospital")),
    "dim_hospital", "hospital_id",
    ["hospital_name", "facility_type", "region", "licensed_beds"], "hospital_key")
seed_unknown("dim_hospital", "hospital_key", "hospital_id", {"hospital_name": "Unknown"})

scd2_merge(
    spark.table(f"{SILVER}.silver_department").select(
        "department_id", "department_name", "service_line", "hospital_id",
        "cost_centre",
        (F.col("is_clinical") == "1").alias("is_clinical"),
        (F.col("is_inpatient_unit") == "1").alias("is_inpatient_unit")),
    "dim_department", "department_id",
    ["department_name", "service_line", "hospital_id", "cost_centre"],
    "department_key")
seed_unknown("dim_department", "department_key", "department_id",
             {"department_name": "Unknown"})

scd2_merge(
    spark.table(f"{SILVER}.silver_bed").select(
        "bed_id", "room_number", "ward_name", "ward_type", "bed_type",
        "hospital_id", "department_id",
        (F.col("is_isolation_capable") == "1").alias("is_isolation_capable"),
        (F.col("is_active") == "1").alias("is_active")),
    "dim_bed", "bed_id",
    ["ward_name", "ward_type", "bed_type", "hospital_id", "department_id",
     "is_active"], "bed_key")
seed_unknown("dim_bed", "bed_key", "bed_id", {"ward_name": "Unknown"})

scd2_merge(
    spark.table(f"{SILVER}.silver_doctor").select(
        "doctor_id", F.col("display_name").alias("doctor_display_name"),
        "specialty", "sub_specialty", "credential", "primary_department_id",
        "hospital_id", "employment_type",
        F.col("fte").cast("decimal(4,2)").alias("fte"),
        F.to_date("hire_date").alias("hire_date"),
        (F.col("is_active") == "1").alias("is_active")),
    "dim_doctor", "doctor_id",
    ["specialty", "sub_specialty", "primary_department_id", "employment_type",
     "fte", "is_active"], "doctor_key")
seed_unknown("dim_doctor", "doctor_key", "doctor_id",
             {"doctor_display_name": "Unknown"})

## 5. Clinical reference dimensions (SCD1)


In [ ]:
scd1_load(
    spark.table(f"{BRONZE}.ref_icd10ca").select(
        "diagnosis_code", "diagnosis_description", "chapter", "category",
        (F.col("is_chronic") == "1").alias("is_chronic")),
    "dim_diagnosis", "diagnosis_key", "diagnosis_code")
seed_unknown("dim_diagnosis", "diagnosis_key", "diagnosis_code",
             {"diagnosis_description": "Unknown"})

scd1_load(
    spark.table(f"{BRONZE}.ref_loinc").select(
        "loinc_code", "test_name", "panel_name", "specimen_type", "result_unit",
        F.col("reference_low").cast("decimal(18,4)").alias("reference_low"),
        F.col("reference_high").cast("decimal(18,4)").alias("reference_high")),
    "dim_lab_test", "lab_test_key", "loinc_code")
seed_unknown("dim_lab_test", "lab_test_key", "loinc_code", {"test_name": "Unknown"})

scd1_load(
    spark.table(f"{BRONZE}.ref_medication").select(
        "din", "generic_name", "brand_name", "atc_code", "atc_class",
        "dosage_form", "route",
        (F.col("is_controlled_substance") == "1").alias("is_controlled_substance"),
        (F.col("is_high_alert") == "1").alias("is_high_alert"),
        (F.col("is_formulary") == "1").alias("is_formulary")),
    "dim_medication", "medication_key", "din")
seed_unknown("dim_medication", "medication_key", "din", {"generic_name": "Unknown"})

scd1_load(
    spark.table(f"{BRONZE}.ref_payer").select(
        "payer_id", "payer_name", "payer_type", "plan_tier"),
    "dim_insurance_provider", "insurance_provider_key", "payer_id")
seed_unknown("dim_insurance_provider", "insurance_provider_key", "payer_id",
             {"payer_name": "Unknown"})


## 6. Small coded dimensions

 Built from ref_code_mapping rather than hardcoded, so adding a discharge
 disposition is a row in a CSV, not a code change.


In [ ]:

cm = spark.table(f"{BRONZE}.ref_code_mapping")

scd1_load(
    (cm.filter(F.col("domain") == "ADMIT_TYPE")
       .select(F.col("standard_code").alias("admission_type_code"),
               F.col("standard_description").alias("admission_type_desc")).distinct()
       .withColumn("is_emergency",
                   F.col("admission_type_code").isin("EMERGENCY", "URGENT"))
       .withColumn("is_elective", F.col("admission_type_code") == "ELECTIVE")),
    "dim_admission_type", "admission_type_key", "admission_type_code")
seed_unknown("dim_admission_type", "admission_type_key", "admission_type_code",
             {"admission_type_desc": "Unknown"})

scd1_load(
    (cm.filter(F.col("domain") == "DISPOSITION")
       .select(F.col("standard_code").alias("disposition_code"),
               F.col("standard_description").alias("disposition_desc")).distinct()
       .withColumn("is_home", F.col("disposition_code").isin("HOME", "HOME_CARE"))
       .withColumn("is_transfer", F.col("disposition_code") == "TRANSFER")
       .withColumn("is_expired", F.col("disposition_code") == "EXPIRED")
       .withColumn("is_against_medical_advice", F.col("disposition_code") == "AMA")),
    "dim_discharge_disposition", "discharge_disposition_key", "disposition_code")
seed_unknown("dim_discharge_disposition", "discharge_disposition_key",
             "disposition_code", {"disposition_desc": "Unknown"})

scd1_load(spark.createDataFrame([
    ("COMPLETED", "Completed", "Attended", True, False, False),
    ("NO_SHOW", "Did not attend", "Missed", False, False, True),
    ("CANCELLED_PATIENT", "Cancelled by patient", "Cancelled", False, True, False),
    ("CANCELLED_PROVIDER", "Cancelled by provider", "Cancelled", False, True, False),
    ("CANCELLED_FACILITY", "Cancelled by facility", "Cancelled", False, True, False),
    ("SCHEDULED", "Scheduled", "Pending", False, False, False),
    ("BOOKED", "Booked", "Pending", False, False, False),
], "status_code string, status_desc string, status_group string, "
   "is_completed boolean, is_cancellation boolean, is_no_show boolean"),
    "dim_appointment_status", "appointment_status_key", "status_code")
seed_unknown("dim_appointment_status", "appointment_status_key", "status_code",
             {"status_desc": "Unknown"})

# X12 835 claim status codes, as they arrive in the CLP segment.
scd1_load(spark.createDataFrame([
    ("1", "Processed as primary", True, True, False),
    ("2", "Processed as secondary", True, True, False),
    ("3", "Processed as tertiary", True, True, False),
    ("4", "Denied", True, False, True),
    ("19", "Processed as primary, forwarded", True, True, False),
    ("22", "Reversal of previous payment", True, False, False),
], "status_code string, status_desc string, is_terminal boolean, "
   "is_approved boolean, is_denied boolean"),
    "dim_claim_status", "claim_status_key", "status_code")
seed_unknown("dim_claim_status", "claim_status_key", "status_code",
             {"status_desc": "Unknown"})

## Summary


In [2]:
print("\n" + "=" * 64)
for t, n, note in summary:
    print(f"{t:<32}{n:>10,}  {note}")
print("=" * 64)

print("\nSpecial members:")
for t, k in [("dim_patient", "patient_key"), ("dim_doctor", "doctor_key"),
             ("dim_hospital", "hospital_key"), ("dim_department", "department_key"),
             ("dim_bed", "bed_key"), ("dim_diagnosis", "diagnosis_key"),
             ("dim_lab_test", "lab_test_key"), ("dim_medication", "medication_key")]:
    print(f"  {t:<28} {spark.table(t).filter(F.col(k) < 0).count()}")

import json
mssparkutils.notebook.exit(json.dumps({
    "batch_id": batch_id, "dimensions": len(summary),
}))

StatementMeta(, b286ab3b-02a5-4bc1-ae59-0133b5ce18b0, 4, Finished, Available, Finished, False)

Gold dimension load GOLD_MANUAL
  dim_date                        1,827 rows
  dim_time                        1,441 rows
  dim_patient                    27,905 rows (initial)
  dim_hospital                        5 rows (initial)
  dim_department                     49 rows (initial)
  dim_bed                           101 rows (initial)
  dim_doctor                        240 rows (initial)
  dim_diagnosis                      32 rows
  dim_lab_test                       25 rows
  dim_medication                     28 rows
  dim_insurance_provider              8 rows
  dim_admission_type                  5 rows
  dim_discharge_disposition           7 rows
  dim_appointment_status              7 rows
  dim_claim_status                    6 rows

dim_patient                         27,905  initial load
dim_hospital                             5  initial load
dim_department                          49  initial load
dim_bed                                101  initial load
dim_doctor    